# Chapter 15 examples: value-based control

This notebook runs the examples of
[Chapter 15](https://thduynguyen.github.io/gtsam/chapter15) on the endless
track of Chapter 3: SARSA, Q-learning, fitted Q iteration and a small
version of DQN. The dynamics table is used only to *sample* transitions,
and to compute the exact answers the samples are checked against. The exact
answers come from the `gtsam/semiring` module, by elimination; the sampling
and the learning updates are plain numpy, since they are what the chapter
is about.

GTSAM Copyright 2010, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/thduynguyen/gtsam/blob/feature/semiringfactor/gtsam/semiring/doc/chapter15_examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    # The semiring module is not in a released GTSAM yet, so install a wheel
    # built from the feature branch.
    %pip install --quiet numpy pyparsing plotly
    %pip install --quiet --no-index --no-deps --find-links https://github.com/thduynguyen/gtsam/releases/expanded_assets/semiring-wheels gtsam-develop
except ImportError:
    pass  # Not running on Colab, do nothing

In [2]:
import numpy as np
from gtsam import DecisionTreeFactor, DiscreteValues, Ordering
from gtsam import SemiringDiscreteFactor, SemiringSum
from gtsam.symbol_shorthand import A, S

np.set_printoptions(precision=4, suppress=True)

## The endless track and its exact values (Section 1)

In [3]:
L, R = 0, 1
gamma = 0.9
prior = np.array([0.5, 0.5, 0.0])  # p(s0)
dynamics = np.zeros((3, 2, 3))  # p(s' | s, a)
dynamics[0, L], dynamics[0, R] = [1, 0, 0], [0.2, 0.8, 0]
dynamics[1, L], dynamics[1, R] = [0.8, 0.2, 0], [0, 0.2, 0.8]
dynamics[2, L], dynamics[2, R] = [0, 0.8, 0.2], [0, 0, 1]
reward = np.array([[0.0, -1.0], [0.0, -1.0], [2.0, 1.0]])  # r(s, a)
coin_flip = np.full((3, 2), 0.5)

The exact values are computed by elimination on one step of the endless
chain, as in Chapters 3 and 4. The discount is a termination outcome: a
fourth state, "ended", reached with probability $1 - \gamma$ after every
move.

In [4]:
def probability(keys, table):
    """Lift a probability table to (p, 0)."""
    return SemiringDiscreteFactor(
        DecisionTreeFactor(keys, np.ravel(table).tolist()))


def value(keys, table):
    """Lift a reward table to (1, r)."""
    return SemiringDiscreteFactor.Reward(
        DecisionTreeFactor(keys, np.ravel(table).tolist()))


def ordering(*keys):
    """An ordering of the given keys."""
    result = Ordering()
    for key in keys:
        result.push_back(key)
    return result


def table(factor, keys):
    """Read a DecisionTreeFactor into an array indexed in the order of keys."""
    result = np.zeros([cardinality for _, cardinality in keys])
    for index in np.ndindex(*result.shape):
        values = DiscreteValues()
        for (key, _), index_of_key in zip(keys, index):
            values[key] = index_of_key
        result[index] = factor(values)
    return result


ENDED = 3
ended_dynamics = np.zeros((4, 2, 4))
ended_dynamics[:3, :, :3] = gamma * dynamics  # continue
ended_dynamics[:3, :, ENDED] = 1 - gamma  # end
ended_dynamics[ENDED, :, ENDED] = 1  # stay ended
ended_reward = np.vstack([reward, [0.0, 0.0]])
ended_prior = np.append(prior, 0.0)

now, move, later = (S(0), 4), (A(0), 2), (S(1), 4)
transition = probability([now, move, later], ended_dynamics)
move_reward = value([now, move], ended_reward)
average, maximum = SemiringSum.Average(), SemiringSum.Maximum()


def policy_factor(policy):
    """A policy table on the three cells, as a factor on (s, a)."""
    return probability([now, move], np.vstack([policy, [0.5, 0.5]]))


def action_values(V):
    """The bucket of the action without a policy factor: (1, Q).

    V is the value of the next state, an array over the four states. The
    next state is eliminated by average, and the reward of the move is added.
    """
    return move_reward * (transition * value([later], V)).sum(ordering(S(1)))


def read_Q(bucket):
    """The table Q(s, a) on the three cells, from the bucket of the action."""
    return table(bucket.value(), [now, move])[:3]


def sum_out_action(bucket, rule, policy=None):
    """Eliminate the action from its bucket; the new value, as an array."""
    if policy is not None:
        bucket = policy_factor(policy) * bucket
    return table(bucket.sum(ordering(A(0)), rule).value(), [now])


def exact_V(policy):
    """V of a stationary policy: eliminate until the value stops changing."""
    V = np.zeros(4)
    while True:
        new_V = sum_out_action(action_values(V), average, policy)
        if np.abs(new_V - V).max() < 1e-13:
            return new_V
        V = new_V


def exact_Q(policy):
    """Q of a stationary policy: the bucket of the action at the fixed point."""
    return read_Q(action_values(exact_V(policy)))


def value_iteration(sweeps):
    """Q after some sweeps of value iteration (Chapter 4), from Q = 0.

    Each sweep is one step of elimination: the next state by average, the
    action by maximum.
    """
    V = np.zeros(4)
    history = []
    for _ in range(sweeps):
        bucket = action_values(V)
        history.append(read_Q(bucket))
        V = sum_out_action(bucket, maximum)
    return history


Q_coin = exact_Q(coin_flip)
Q_star = value_iteration(500)[-1]
print("Q of the coin flip =\n", Q_coin)
print("Q* =\n", Q_star)
assert np.allclose(Q_coin, [[-0.2023, -0.2472], [0.0365, 2.1669],
                            [3.5354, 4.7108]], atol=1e-4)
assert np.allclose(Q_star, [[4.8775, 5.4194], [5.2629, 7.5610],
                            [9.2439, 10.0]], atol=1e-4)

Q of the coin flip =
 [[-0.2023 -0.2472]
 [ 0.0365  2.1669]
 [ 3.5354  4.7108]]
Q* =
 [[ 4.8775  5.4194]
 [ 5.2629  7.561 ]
 [ 9.2439 10.    ]]


## A simulator: the dynamics factor through samples (Section 1)

From here on the algorithms never read the table `dynamics`. They call
`step`, which returns one sampled next state. After each move the episode
ends with probability $1 - \gamma$ (Chapter 3), and the robot is put back
at the start.

In [5]:
def step(rng, s, a):
    """One sampled transition: the reward and the next state."""
    return reward[s, a], rng.choice(3, p=dynamics[s, a])


def restart(rng, s_next):
    """The state the robot acts in next: s' if the episode continues."""
    return s_next if rng.random() < gamma else rng.choice(3, p=prior)


def collect(rng, policy, count):
    """A stream of transitions (s, a, r, s') under a policy."""
    data = np.zeros((count, 4))
    s = rng.choice(3, p=prior)
    for i in range(count):
        a = rng.choice(2, p=policy[s])
        r, s_next = step(rng, s, a)
        data[i] = s, a, r, s_next
        s = restart(rng, s_next)
    return data


rng = np.random.default_rng(0)
data = collect(rng, coin_flip, 200_000)
counts = np.zeros((3, 2))
np.add.at(counts, (data[:, 0].astype(int), data[:, 1].astype(int)), 1)
print("first transitions (s, a, r, s'):\n", data[:5])
print("share of each (s, a) in the data =\n", counts / counts.sum())


def discounted_visitation(policy):
    """The forward message d of Chapter 3, by eliminating forward in time.

    One step sums out the current state and action and leaves the marginal of
    the next state. The mass still in the three cells after t moves is
    gamma^t d_t, and d is its sum over t.
    """
    marginal, d = ended_prior, np.zeros(3)
    while marginal[:3].sum() > 1e-13:
        d += marginal[:3]
        joint = probability([now], marginal) * policy_factor(policy) * transition
        marginal = table(joint.sum(ordering(S(0), A(0))).probability(), [later])
    return d


d = discounted_visitation(coin_flip)
print("discounted visitation d =", d)
print("d / 10, split over the two moves =", d / d.sum() / 2)
assert np.allclose(d, [3.8062, 3.4746, 2.7192], atol=1e-4)
assert np.allclose(counts / counts.sum(), (d / d.sum() / 2)[:, None],
                   atol=0.001)

first transitions (s, a, r, s'):
 [[ 1.  0.  0.  0.]
 [ 0.  1. -1.  1.]
 [ 1.  1. -1.  2.]
 [ 1.  0.  0.  1.]
 [ 1.  1. -1.  1.]]
share of each (s, a) in the data =
 [[0.1905 0.19  ]
 [0.1741 0.174 ]
 [0.136  0.1354]]
discounted visitation d = [3.8062 3.4746 2.7192]
d / 10, split over the two moves = [0.1903 0.1737 0.136 ]


## SARSA and Q-learning (Section 2)

Both replay the same stream of coin-flip transitions. They differ in one
line: the value used for the next state.

In [6]:
def td_control(data, rule, rng, exponent=0.6):
    """Tabular TD on a stream of transitions, with a decaying step size."""
    Q = np.zeros((3, 2))
    visits = np.zeros((3, 2))
    for s, a, r, s_next in data:
        s, a, s_next = int(s), int(a), int(s_next)
        if rule == "sarsa":  # the next action of the coin flip, sampled
            future = Q[s_next, rng.choice(2)]
        elif rule == "expected sarsa":  # the average under the coin flip
            future = coin_flip[s_next] @ Q[s_next]
        else:  # Q-learning: the maximum
            future = Q[s_next].max()
        visits[s, a] += 1
        alpha = visits[s, a] ** -exponent
        Q[s, a] += alpha * (r + gamma * future - Q[s, a])
    return Q


rng = np.random.default_rng(1)
Q_sarsa = td_control(data, "sarsa", rng)
Q_expected = td_control(data, "expected sarsa", rng)
Q_learning = td_control(data, "q-learning", rng)
print("SARSA =\n", Q_sarsa)
print("largest error against Q of the coin flip:",
      np.abs(Q_sarsa - Q_coin).max())
print("expected SARSA: largest error",
      np.abs(Q_expected - Q_coin).max())
print("Q-learning =\n", Q_learning)
print("largest error against Q*:", np.abs(Q_learning - Q_star).max())
print("greedy policy of Q-learning (0 = L, 1 = R):", Q_learning.argmax(axis=1))
assert np.abs(Q_sarsa - Q_coin).max() < 0.15
assert np.abs(Q_expected - Q_coin).max() < 0.1
assert np.abs(Q_learning - Q_star).max() < 0.1
assert list(Q_learning.argmax(axis=1)) == [R, R, R]

SARSA =
 [[-0.2359 -0.3057]
 [ 0.0101  2.1004]
 [ 3.4277  4.6465]]
largest error against Q of the coin flip: 0.10768405228356981
expected SARSA: largest error 0.07100286307675763
Q-learning =
 [[ 4.8428  5.3943]
 [ 5.2261  7.5423]
 [ 9.1666 10.    ]]
largest error against Q*: 0.07728041131846375
greedy policy of Q-learning (0 = L, 1 = R): [1 1 1]


## SARSA as control (Section 3)

Stage 2 is greedy: after every update the policy becomes "the best move of
the current table, except for a random move 10% of the time", and the next
sample is drawn from that policy.

In [7]:
def sarsa_control(rng, count, explore=0.1, exponent=0.6):
    Q, visits = np.zeros((3, 2)), np.zeros((3, 2))

    def act(s):
        if rng.random() < explore:
            return rng.choice(2)
        return int(Q[s].argmax())

    s = rng.choice(3, p=prior)
    a = act(s)
    for _ in range(count):
        r, s_next = step(rng, s, a)
        a_next = act(s_next)
        visits[s, a] += 1
        Q[s, a] += visits[s, a] ** -exponent * (
            r + gamma * Q[s_next, a_next] - Q[s, a])
        if rng.random() < gamma:  # the episode continues
            s, a = s_next, a_next
        else:  # it ends: back to the start
            s = rng.choice(3, p=prior)
            a = act(s)
    return Q


def epsilon_greedy(Q, explore=0.1):
    policy = np.full((3, 2), explore / 2)
    policy[np.arange(3), Q.argmax(axis=1)] += 1 - explore
    return policy


Q_control = sarsa_control(np.random.default_rng(2), 200_000)
Q_target = exact_Q(epsilon_greedy(Q_control))
print("SARSA with a greedy Stage 2 =\n", Q_control)
print("exact Q of its own policy =\n", Q_target)
print("greedy moves:", Q_control.argmax(axis=1))
assert list(Q_control.argmax(axis=1)) == [R, R, R]
assert np.abs(Q_control - Q_target).max() < 0.25

SARSA with a greedy Stage 2 =
 [[4.4439 5.0374]
 [4.9348 7.2123]
 [8.8482 9.6507]]
exact Q of its own policy =
 [[4.4505 4.971 ]
 [4.8306 7.174 ]
 [8.8069 9.6297]]
greedy moves: [1 1 1]


## Importance weights (Section 4)

The data come from the coin flip $\pi_D$; the policy to evaluate, $\pi$,
moves Right with probability 0.9. All averages below are computed exactly,
by elimination, so the differences are not sampling noise.

In [8]:
target_policy = np.tile([0.1, 0.9], (3, 1))
V_pi = exact_V(target_policy)
Q_pi = read_Q(action_values(V_pi))

# The inner bucket holds the value Q of pi for the pair (s', a').
inner = action_values(V_pi)

# One step: r + gamma * (average of Q under pi at s'). The next action is
# averaged under pi itself, so the data policy plays no part.
one_step = read_Q(action_values(sum_out_action(inner, average, target_policy)))

# Two steps: r + gamma r' + gamma^2 V(s''). Now the next action a' is the one
# in the data, so it is averaged under the data policy.
two_step_plain = read_Q(action_values(
    sum_out_action(inner, average, coin_flip)))

# The importance weight is one more factor on (s', a'): multiplied with the
# factor of the data policy, it gives the factor of pi.
rho = target_policy / coin_flip  # pi(a' | s') / pi_D(a' | s')
weight = probability([now, move], np.vstack([rho, [1.0, 1.0]]))
two_step_weighted = read_Q(action_values(
    sum_out_action(weight * inner, average, coin_flip)))
print("Q of pi =\n", Q_pi)
print("one-step target, averaged =\n", one_step)
print("two-step target, averaged, no weights =\n", two_step_plain)
print("two-step target, averaged, with weights =\n", two_step_weighted)
print("importance weights rho =\n", rho)
assert np.allclose(Q_pi, [[3.979, 4.470], [4.352, 6.730], [8.315, 9.202]],
                   atol=1e-3)
assert np.allclose(one_step, Q_pi)
assert np.allclose(two_step_plain,
                   [[3.802, 3.750], [4.039, 6.303], [7.566, 8.882]], atol=1e-3)
assert np.allclose(two_step_weighted, Q_pi)
assert np.abs(two_step_plain - Q_pi).max() > 0.5

Q of pi =
 [[3.9789 4.4701]
 [4.3517 6.7299]
 [8.3146 9.2017]]
one-step target, averaged =
 [[3.9789 4.4701]
 [4.3517 6.7299]
 [8.3146 9.2017]]
two-step target, averaged, no weights =
 [[3.802  3.7498]
 [4.039  6.3032]
 [7.5658 8.8823]]
two-step target, averaged, with weights =
 [[3.9789 4.4701]
 [4.3517 6.7299]
 [8.3146 9.2017]]
importance weights rho =
 [[0.2 1.8]
 [0.2 1.8]
 [0.2 1.8]]


## Fitted Q iteration (Section 5)

A batch with five transitions per pair $(s, a)$, whose next states occur in
exactly the proportions of the dynamics table. With a table as the
regressor, each fit is one sweep of value iteration.

In [9]:
batch = []
for s in range(3):
    for a in range(2):
        for s_next in range(3):
            for _ in range(int(round(5 * dynamics[s, a, s_next]))):
                batch.append((s, a, reward[s, a], s_next))
batch = np.array(batch)
print("batch size:", len(batch))


def fit_table(batch, targets):
    """Least squares with one parameter per (s, a): the mean target."""
    total, count = np.zeros((3, 2)), np.zeros((3, 2))
    index = (batch[:, 0].astype(int), batch[:, 1].astype(int))
    np.add.at(total, index, targets)
    np.add.at(count, index, 1)
    return total / count


def fitted_q_iteration(batch, sweeps):
    Q = np.zeros((3, 2))
    history = []
    for _ in range(sweeps):
        targets = batch[:, 2] + gamma * Q[batch[:, 3].astype(int)].max(axis=1)
        Q = fit_table(batch, targets)  # the regression
        history.append(Q)
    return history


fitted = fitted_q_iteration(batch, 200)
exact = value_iteration(200)
for k in [1, 2, 5, 100]:
    print(f"sweep {k:3d}: V = {fitted[k - 1].max(axis=1)}")
    assert np.allclose(fitted[k - 1], exact[k - 1])
assert np.allclose(fitted[-1], Q_star, atol=1e-4)

# The same on 2000 sampled transitions: value iteration on the empirical model.
sample = data[:2000]
Q_fitted = fitted_q_iteration(sample, 200)[-1]
print("fitted Q iteration on 2000 samples =\n", Q_fitted)
print("largest error against Q*:", np.abs(Q_fitted - Q_star).max())
assert np.abs(Q_fitted - Q_star).max() < 0.5
assert list(Q_fitted.argmax(axis=1)) == [R, R, R]

batch size: 30
sweep   1: V = [0. 0. 2.]
sweep   2: V = [0.   0.44 2.8 ]
sweep   5: V = [0.2467 2.3126 4.7512]
sweep 100: V = [5.4192 7.5607 9.9998]
fitted Q iteration on 2000 samples =
 [[ 4.7759  5.3065]
 [ 5.2114  7.5239]
 [ 9.2337 10.    ]]
largest error against Q*: 0.11284461827737591


## A small DQN (Section 6)

The three ingredients of DQN, with a table in place of the network: a
replay buffer, minibatch gradient steps on the squared residual, and a
frozen copy of the table for the targets, refreshed every 100 steps.

In [10]:
def small_dqn(rng, steps, batch_size=32, step_size=0.1, refresh=100,
              explore=0.2, capacity=5000):
    Q = np.zeros((3, 2))  # theta_Q: here one parameter per (s, a)
    Q_frozen = Q.copy()  # theta_Q^-: the frozen copy
    buffer = np.zeros((capacity, 4))  # the replay buffer D
    s = rng.choice(3, p=prior)
    for k in range(steps):
        # Act: mostly greedy, sometimes random; store the transition.
        a = rng.choice(2) if rng.random() < explore else int(Q[s].argmax())
        r, s_next = step(rng, s, a)
        buffer[k % capacity] = s, a, r, s_next  # overwrite the oldest
        s = restart(rng, s_next)
        # Learn: a minibatch of old transitions, targets from the frozen copy.
        chosen = buffer[rng.integers(min(k + 1, capacity), size=batch_size)]
        b_s, b_a = chosen[:, 0].astype(int), chosen[:, 1].astype(int)
        targets = chosen[:, 2] + gamma * Q_frozen[
            chosen[:, 3].astype(int)].max(axis=1)
        gradient = np.zeros((3, 2))
        np.add.at(gradient, (b_s, b_a), targets - Q[b_s, b_a])
        Q += step_size * gradient / batch_size
        if (k + 1) % refresh == 0:
            Q_frozen = Q.copy()
    return Q


Q_dqn = small_dqn(np.random.default_rng(3), 30_000)
print("small DQN =\n", Q_dqn)
print("largest error against Q*:", np.abs(Q_dqn - Q_star).max())
print("greedy moves:", Q_dqn.argmax(axis=1))
assert np.abs(Q_dqn - Q_star).max() < 0.5
assert list(Q_dqn.argmax(axis=1)) == [R, R, R]

small DQN =
 [[ 4.8104  5.3383]
 [ 5.172   7.6387]
 [ 9.1599 10.    ]]
largest error against Q*: 0.09091571856395841
greedy moves: [1 1 1]
